# 14. Serie 2: Viajeros por vía aérea

In [ ]:
descomposicion_aerea = seasonal_decompose(
    aerea["Viajero"],
    model="additive",
    period=12
)

fig = descomposicion_aerea.plot()
fig.set_size_inches(14,10)

plt.tight_layout()
plt.show()

## Interpretación de la descomposición

La descomposición de la serie correspondiente a los viajeros que ingresan por vía aérea permite identificar claramente los componentes de tendencia, estacionalidad y residuales.

### Tendencia

El componente de tendencia muestra un crecimiento gradual desde el año 2009 hasta principios de 2020, reflejando el incremento sostenido del ingreso de viajeros por vía aérea durante ese período. Sin embargo, a partir de 2020 se observa una caída muy pronunciada provocada por la pandemia de COVID-19 y las restricciones impuestas al transporte aéreo internacional. Posteriormente, durante los años 2021 y 2022, la tendencia presenta una recuperación progresiva, aunque sin alcanzar completamente el crecimiento continuo observado antes de la pandemia.

### Estacionalidad

El componente estacional presenta un patrón que se repite de forma consistente cada doce meses, evidenciando una estacionalidad anual bien definida. Esto indica que existen meses en los que el ingreso de viajeros por vía aérea aumenta de forma recurrente, mientras que en otros disminuye, comportamiento asociado a temporadas vacacionales y períodos de mayor actividad turística.

### Componente residual

Los residuos permanecen cercanos a cero durante la mayor parte del período analizado, lo que indica que la tendencia y la estacionalidad explican gran parte de la variabilidad de la serie. No obstante, durante el año 2020 se observan residuos de mayor magnitud, tanto positivos como negativos, reflejando el efecto extraordinario de la pandemia sobre el transporte aéreo.

### Conclusión

La descomposición confirma que la serie presenta una tendencia de largo plazo y una estacionalidad anual claramente definida. Además, evidencia un cambio estructural importante durante la pandemia, el cual modifica temporalmente el comportamiento de la serie. Debido a estas características, será necesario analizar formalmente la estacionariedad antes de construir los modelos de pronóstico.

In [ ]:
resultado_adf_aerea = adfuller(aerea["Viajero"])

print("Estadístico ADF:", resultado_adf_aerea[0])
print("Valor p:", resultado_adf_aerea[1])

print("\nValores críticos:")
for key, value in resultado_adf_aerea[4].items():
    print(f"{key}: {value:.4f}")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

plot_acf(aerea["Viajero"], lags=40, ax=ax[0])
plot_pacf(aerea["Viajero"], lags=40, ax=ax[1], method="ywm")

ax[0].set_title("ACF - Serie aérea")
ax[1].set_title("PACF - Serie aérea")

plt.tight_layout()
plt.show()

## Interpretación de la prueba de estacionariedad

Para evaluar la estacionariedad de la serie correspondiente a los viajeros que ingresan por vía aérea se aplicó la prueba Dickey-Fuller Aumentada (ADF), además del análisis de las funciones de autocorrelación (ACF) y autocorrelación parcial (PACF).

### Prueba Dickey-Fuller Aumentada (ADF)

Los resultados obtenidos fueron:

- **Estadístico ADF:** -3.1939
- **Valor p:** 0.0203

Dado que el valor p es menor que 0.05, se rechaza la hipótesis nula de que la serie presenta una raíz unitaria. Esto indica que, estadísticamente, la serie puede considerarse estacionaria con un nivel de significancia del 5 %.

Sin embargo, al igual que en la serie del total de viajeros, el análisis visual muestra una tendencia de crecimiento antes del año 2020 y un cambio estructural importante ocasionado por la pandemia de COVID-19. Por ello, aunque la prueba ADF sugiere estacionariedad, estos cambios deben tenerse en cuenta durante la construcción de los modelos de pronóstico.

### Función de Autocorrelación (ACF)

La función de autocorrelación presenta valores positivos elevados en los primeros rezagos, los cuales disminuyen gradualmente conforme aumenta el número de rezagos. Este comportamiento indica que las observaciones mantienen una fuerte dependencia temporal y que los valores actuales están relacionados con los meses anteriores.

También se observa un patrón compatible con una estacionalidad anual, ya que la autocorrelación permanece significativa durante varios rezagos.

### Función de Autocorrelación Parcial (PACF)

La PACF muestra un pico muy pronunciado en el primer rezago y algunos picos adicionales de menor magnitud en rezagos posteriores. Después de estos primeros rezagos, la mayoría de las autocorrelaciones parciales permanecen dentro de los límites de confianza, indicando que la influencia directa disminuye conforme aumenta el rezago.

Este comportamiento sugiere la presencia de componentes autorregresivos y estacionales, lo que hace apropiado utilizar un modelo ARIMA para describir la dinámica de la serie.

### Conclusión

Los resultados de la prueba ADF, junto con el análisis de las funciones ACF y PACF, indican que la serie presenta una estructura temporal adecuada para ser modelada mediante un modelo ARIMA. Aunque la prueba estadística rechaza la presencia de una raíz unitaria, el comportamiento observado durante la pandemia representa un cambio estructural importante que deberá considerarse al evaluar el desempeño del modelo.

# 16. Modelo ARIMA para la serie aérea

In [ ]:
# División entrenamiento y prueba

n = len(aerea)

train_size = int(n * 0.70)

train_aerea = aerea["Viajero"][:train_size]
test_aerea = aerea["Viajero"][train_size:]

print(len(train_aerea))
print(len(test_aerea))

In [ ]:
modelo_auto_aerea = auto_arima(
    train_aerea,
    seasonal=True,
    m=12,
    trace=True,
    stepwise=True,
    suppress_warnings=True
)

print(modelo_auto_aerea.summary())

# 16. Modelo ARIMA para la serie aérea

Para la serie correspondiente a los viajeros que ingresan por vía aérea se utilizó la función `auto_arima` con el objetivo de seleccionar automáticamente el modelo que mejor representara el comportamiento de la serie. La selección se realizó utilizando el criterio de información de Akaike (AIC), eligiendo el modelo con el menor valor entre todas las combinaciones evaluadas.

El modelo seleccionado fue:

**ARIMA(3,0,2)(2,0,0)[12]**

Este modelo incorpora tres términos autorregresivos, dos términos de media móvil y dos componentes autorregresivos estacionales con un período de 12 meses.

## Interpretación del modelo ARIMA

Después de evaluar múltiples configuraciones, el algoritmo seleccionó el modelo **ARIMA(3,0,2)(2,0,0)[12]**, ya que presentó el menor valor de AIC (**3217.696**) entre todos los modelos analizados.

A diferencia de la serie del total de viajeros, este modelo no requiere diferenciación (**d = 0**), lo cual es consistente con los resultados obtenidos en la prueba Dickey-Fuller Aumentada, donde se concluyó que la serie podía considerarse estacionaria.

Los coeficientes autorregresivos y de media móvil presentan, en su mayoría, valores p inferiores a 0.05, indicando que son estadísticamente significativos y contribuyen al ajuste del modelo. Asimismo, los componentes estacionales (`ar.S.L12` y `ar.S.L24`) también resultan significativos, confirmando la existencia de una estacionalidad anual en la serie.

El valor de **AIC = 3217.696** es considerablemente menor que el obtenido para la serie del total de viajeros, lo que sugiere que la dinámica de la serie aérea puede representarse de manera más eficiente mediante un modelo ARIMA.

En general, el modelo seleccionado describe adecuadamente la dependencia temporal y la estacionalidad presentes en la serie, por lo que constituye un buen candidato para realizar las predicciones y compararlo posteriormente con el modelo Holt-Winters.

In [ ]:
# Ajustar el modelo seleccionado

modelo_aerea = modelo_auto_aerea.fit(train_aerea)

residuos_aerea = pd.Series(modelo_aerea.resid())

In [ ]:
plt.figure(figsize=(14,5))

plt.plot(residuos_aerea)

plt.title("Residuos del modelo ARIMA - Serie aérea")

plt.xlabel("Observaciones")
plt.ylabel("Error")

plt.show()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,5))

residuos_aerea.hist(
    bins=25,
    ax=ax[0]
)

ax[0].set_title("Distribución de residuos")

plot_acf(
    residuos_aerea,
    lags=30,
    ax=ax[1]
)

ax[1].set_title("ACF de residuos")

plt.tight_layout()

plt.show()

## Interpretación del análisis de residuos

Una vez ajustado el modelo ARIMA para la serie de viajeros por vía aérea, se analizó el comportamiento de los residuos con el propósito de verificar si el modelo logró capturar adecuadamente la estructura de la serie.

### Gráfico de residuos

En el gráfico de residuos se observa que la mayoría de los errores se distribuyen alrededor de cero sin presentar una tendencia definida. Esto indica que el modelo explica gran parte del comportamiento de la serie. Sin embargo, durante el año 2020 se identifican algunos residuos de mayor magnitud, asociados al impacto excepcional que tuvo la pandemia de COVID-19 sobre el transporte aéreo internacional.

### Distribución de residuos

El histograma muestra que la mayor parte de los residuos se concentra alrededor de cero, aunque existen algunos valores extremos que generan una ligera asimetría en la distribución. Estos valores corresponden principalmente al período de la pandemia, cuando el comportamiento de la serie presentó cambios abruptos difíciles de modelar.

### Autocorrelación de los residuos

La función de autocorrelación (ACF) muestra que prácticamente todos los coeficientes permanecen dentro de los límites de confianza, lo que indica que no existe autocorrelación significativa en los residuos. Esto sugiere que el modelo logró capturar la dependencia temporal presente en la serie y que los errores se comportan de forma similar a un ruido blanco.

### Conclusión

En general, el modelo ARIMA presenta un buen ajuste para la serie de viajeros por vía aérea. Los residuos no muestran patrones sistemáticos ni autocorrelaciones importantes, por lo que puede concluirse que el modelo representa adecuadamente el comportamiento histórico de la serie. Las principales desviaciones corresponden al período de la pandemia, un evento extraordinario que afectó significativamente el ingreso de viajeros por vía aérea.

# 17. Predicción con el modelo ARIMA

In [ ]:
pred_aerea = modelo_aerea.predict(
    n_periods=len(test_aerea)
)

pred_aerea = pd.Series(
    pred_aerea,
    index=test_aerea.index
)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(train_aerea, label="Entrenamiento")
plt.plot(test_aerea, label="Valores reales")
plt.plot(pred_aerea, label="Predicción ARIMA")

plt.title("Predicción ARIMA - Serie aérea")

plt.xlabel("Fecha")
plt.ylabel("Viajeros")

plt.legend()

plt.show()

In [ ]:
mae_aerea = mean_absolute_error(
    test_aerea,
    pred_aerea
)

rmse_aerea = np.sqrt(
    mean_squared_error(
        test_aerea,
        pred_aerea
    )
)

print(f"MAE : {mae_aerea:,.2f}")
print(f"RMSE: {rmse_aerea:,.2f}")

print(f"AIC : {modelo_aerea.aic():.2f}")
print(f"BIC : {modelo_aerea.bic():.2f}")

## Interpretación de las predicciones del modelo ARIMA

El modelo ARIMA fue utilizado para generar predicciones sobre el conjunto de prueba correspondiente a la serie de viajeros por vía aérea.

En la gráfica se observa que el modelo logra seguir de manera razonable el comportamiento general de la serie. Aunque las predicciones tienden a subestimar algunos de los picos más altos observados después de la recuperación del turismo, el modelo consigue representar adecuadamente la tendencia general y las variaciones estacionales presentes en los datos.

Las métricas obtenidas fueron:

- **MAE:** 29,072.23
- **RMSE:** 36,646.44
- **AIC:** 3217.70
- **BIC:** 3244.61

Los valores de MAE y RMSE son considerablemente menores que los obtenidos para la serie del total de viajeros, lo que indica que el modelo presenta una mejor capacidad de predicción para la serie aérea.

Este mejor desempeño puede explicarse porque la serie de viajeros por vía aérea presenta un comportamiento más uniforme y menos influenciado por factores externos distintos al transporte aéreo. Aunque la pandemia produjo una disminución importante en el número de viajeros, la recuperación posterior muestra un patrón más estable que facilita su modelado mediante un modelo ARIMA.

En conclusión, el modelo ARIMA ofrece un buen ajuste para esta serie y demuestra una capacidad predictiva satisfactoria sobre el conjunto de prueba, convirtiéndose en una alternativa adecuada para realizar pronósticos del ingreso de viajeros por vía aérea.

# 18. Modelo Holt-Winters para la serie aérea

In [ ]:
modelo_hw_aerea = ExponentialSmoothing(
    train_aerea,
    trend="add",
    seasonal="add",
    seasonal_periods=12
).fit()

In [ ]:
pred_hw_aerea = modelo_hw_aerea.forecast(len(test_aerea))

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(train_aerea, label="Entrenamiento")
plt.plot(test_aerea, label="Valores reales")
plt.plot(pred_hw_aerea, label="Predicción Holt-Winters")

plt.title("Predicción Holt-Winters - Serie aérea")

plt.xlabel("Fecha")
plt.ylabel("Viajeros")

plt.legend()

plt.show()

In [ ]:
mae_hw_aerea = mean_absolute_error(test_aerea, pred_hw_aerea)

rmse_hw_aerea = np.sqrt(
    mean_squared_error(test_aerea, pred_hw_aerea)
)

print(f"MAE : {mae_hw_aerea:,.2f}")
print(f"RMSE: {rmse_hw_aerea:,.2f}")

## Interpretación del modelo Holt-Winters

El modelo Holt-Winters fue aplicado a la serie de viajeros por vía aérea con el propósito de comparar su desempeño frente al modelo ARIMA.

En la gráfica se observa que el modelo logra seguir parcialmente la tendencia del conjunto de prueba durante los primeros meses; sin embargo, conforme avanza el horizonte de predicción comienza a subestimar el número de viajeros. Las predicciones muestran una tendencia descendente que no refleja el comportamiento real de la serie, especialmente durante el período de recuperación posterior a la pandemia.

Las métricas obtenidas fueron:

- **MAE:** 58,261.62
- **RMSE:** 62,865.99

Aunque Holt-Winters incorpora componentes de tendencia y estacionalidad, el modelo no consigue adaptarse completamente al cambio estructural provocado por la pandemia ni a la recuperación observada en los años posteriores. Como consecuencia, presenta errores mayores que los obtenidos con el modelo ARIMA.

En comparación con ARIMA, Holt-Winters ofrece un desempeño inferior para la serie aérea, por lo que no fue seleccionado como el modelo más adecuado para realizar los pronósticos.

# 19. Comparación de modelos para la serie aérea

Se compararon los modelos ARIMA y Holt-Winters utilizando las métricas MAE y RMSE para determinar cuál presenta un mejor desempeño sobre el conjunto de prueba.

| Modelo | MAE | RMSE |
|---------|------------:|------------:|
| **ARIMA** | **29,072.23** | **36,646.44** |
| Holt-Winters | 58,261.62 | 62,865.99 |

El modelo ARIMA obtuvo los menores valores tanto de MAE como de RMSE, lo que indica una mejor capacidad para representar el comportamiento de la serie y generar predicciones más cercanas a los valores reales.

Por lo tanto, el modelo **ARIMA** fue seleccionado como el mejor método de pronóstico para la serie correspondiente a los viajeros que ingresan por vía aérea.

# 20. Comparación general de las series analizadas

Como parte del laboratorio se analizaron dos series de tiempo:

1. Total mensual de viajeros internacionales.
2. Viajeros que ingresan por vía aérea.

En ambas series se observó una tendencia creciente antes del año 2020, una disminución abrupta ocasionada por la pandemia de COVID-19 y una recuperación progresiva durante los años posteriores. Asimismo, ambas presentan un patrón estacional anual claramente definido.

Sin embargo, el comportamiento de la serie aérea resultó más estable y homogéneo, permitiendo obtener un mejor ajuste mediante el modelo ARIMA. En contraste, la serie del total de viajeros mostró una mayor variabilidad debido a que incorpora el comportamiento conjunto de todas las vías de ingreso, lo que dificultó el proceso de predicción.

En ambas series el modelo ARIMA presentó un mejor desempeño que Holt-Winters, obteniendo menores errores de predicción.


# 21. Conclusiones

El análisis exploratorio permitió identificar que el conjunto de datos presenta una buena calidad, ya que no contiene valores faltantes ni registros duplicados. Además, se observó que el ingreso de viajeros internacionales presenta una marcada estacionalidad y una fuerte afectación durante el año 2020 como consecuencia de la pandemia de COVID-19.

La descomposición de ambas series confirmó la existencia de una tendencia de largo plazo y un patrón estacional anual. Asimismo, las pruebas Dickey-Fuller, junto con las funciones de autocorrelación y autocorrelación parcial, permitieron evaluar la estructura temporal de cada serie y justificar el uso de modelos ARIMA.

Para ambas series se construyeron modelos ARIMA y Holt-Winters. Los resultados mostraron que el modelo ARIMA obtuvo un mejor desempeño en los dos casos, presentando menores valores de MAE y RMSE respecto a Holt-Winters.

Finalmente, se concluye que el modelo ARIMA constituye la alternativa más adecuada para realizar pronósticos sobre las series analizadas. No obstante, el cambio estructural ocasionado por la pandemia representa un desafío importante para cualquier modelo de series de tiempo, ya que introduce variaciones que no pueden explicarse únicamente mediante la información histórica disponible.
